# 2.2 — Geração de Embeddings e Banco Vetorial

## Objetivo

Este notebook tem como objetivo gerar, armazenar e disponibilizar embeddings das avaliações de clientes da Olist para utilização em um sistema de **Voice of Customer Intelligence com RAG (Retrieval-Augmented Generation)**.

O modelo utilizado é o `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`, selecionado no notebook **2.1 — Comparação de Modelos de Embeddings** após a avaliação de diferentes abordagens de recuperação semântica.

## Metodologia

O processamento está organizado em quatro etapas:

1. **Geração dos embeddings:** transformação das avaliações textuais em vetores numéricos normalizados de 384 dimensões.
2. **Persistência dos dados:** armazenamento dos embeddings e seus metadados em uma tabela Delta no Databricks.
3. **Indexação e validação:** utilização do Databricks Vector Search para recuperação semântica e aplicação de filtros por metadados.
4. **Integração com RAG:** disponibilização das configurações necessárias para consulta ao índice vetorial pelos demais componentes da solução.

## Dados utilizados

- **Origem:** `workspace.raw.f4_reviews_rag`
- **Quantidade de avaliações:** 40.739
- **Modelo de embeddings:** `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`
- **Tabela de destino:** `workspace.raw.f4_reviews_embeddings`
- **Índice vetorial:** `workspace.raw.f4_reviews_vector_index`

## Resultado esperado

Ao final, teremos uma base vetorial persistente, indexada e validada, capaz de recuperar avaliações semanticamente relevantes e aplicar filtros por metadados.

Essa estrutura será utilizada como mecanismo de recuperação de informações no pipeline RAG desenvolvido pelo grupo.

## 1. Geração dos embeddings

Nesta etapa, utilizamos o modelo `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`, selecionado no notebook 2.1, para transformar as avaliações textuais da Olist em representações vetoriais.

O processamento contempla:

1. Preparação do ambiente e carregamento do modelo.
2. Leitura das avaliações da tabela `workspace.raw.f4_reviews_rag`.
3. Geração de embeddings normalizados de 384 dimensões.
4. Preparação dos vetores e metadados para armazenamento no Delta Lake.

A base contém **40.739 avaliações**, processadas em lotes de 32 registros.

Os embeddings gerados serão utilizados posteriormente para criar o índice de busca vetorial no Databricks.

In [0]:
%pip install sentence-transformers

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from sentence_transformers import SentenceTransformer
import numpy as np
import time

MODELO = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
TABELA_ORIGEM = "workspace.raw.f4_reviews_rag"
TABELA_DESTINO = "workspace.raw.f4_reviews_embeddings"

modelo = SentenceTransformer(MODELO)

df = spark.table(TABELA_ORIGEM)
dados = df.toPandas().reset_index(drop=True)

print(f"Avaliações carregadas: {len(dados):,}")

inicio = time.time()

vetores = modelo.encode(
    dados["review_full_text"].astype(str).tolist(),
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

tempo = time.time() - inicio

print(f"Embeddings gerados: {len(vetores):,}")
print(f"Dimensões: {vetores.shape[1]}")
print(f"Tempo: {tempo:.2f} segundos")

# Identificador único e conversão dos vetores para armazenamento no Spark
dados["embedding_id"] = [
    str(i) for i in range(1, len(dados) + 1)
]

dados["embedding"] = [
    vetor.astype(np.float32).tolist()
    for vetor in vetores
]

print("Dados preparados para armazenamento.")

/local_disk0/.ephemeral_nfs/envs/pythonEnv-9298cb08-6c66-4904-9e2e-e4c2f438ea81/lib/python3.12/site-packages/torch/_vmap_internals.py:9: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  from torch.utils._pytree import _broadcast_to_and_flatten, tree_flatten, tree_unflatten


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

... (7 outputs left out)

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Avaliações carregadas: 40,739


Batches:   0%|          | 0/1274 [00:00<?, ?it/s]

Embeddings gerados: 40,739
Dimensões: 384
Tempo: 702.09 segundos
Dados preparados para armazenamento.


## 2. Persistência e validação dos embeddings

Após a geração dos embeddings, os vetores e os metadados das avaliações são armazenados em uma tabela Delta no Databricks.

Essa etapa tem como objetivos:

1. Definir o esquema de armazenamento, preservando as colunas originais e adicionando `embedding_id` e `embedding`.
2. Persistir os 40.739 registros na tabela `workspace.raw.f4_reviews_embeddings`.
3. Validar a quantidade de registros armazenados e verificar exemplos dos dados.

A persistência permite reutilizar os embeddings sem executar novamente o modelo de linguagem, além de disponibilizar a tabela para indexação no Databricks Vector Search.

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    ArrayType,
    FloatType
)

# Preserva o esquema original e adiciona os campos necessários ao índice vetorial
schema_destino = StructType(
    list(df.schema.fields) + [
        StructField("embedding_id", StringType(), False),
        StructField("embedding", ArrayType(FloatType()), True)
    ]
)

colunas = df.columns + ["embedding_id", "embedding"]

dados_delta = dados[colunas].copy()

df_embeddings = spark.createDataFrame(
    dados_delta,
    schema=schema_destino
)

# Gravação inicial da tabela Delta
df_embeddings.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABELA_DESTINO)

print("Tabela Delta criada com sucesso!")

# Validação dos dados persistidos
df_validacao = spark.table(TABELA_DESTINO)

print(f"Registros persistidos: {df_validacao.count():,}")

display(
    df_validacao.select(
        "embedding_id",
        "review_id",
        "review_full_text",
        "embedding"
    ).limit(5)
)

Tabela Delta criada com sucesso!
Registros persistidos: 40,739


embedding_id review_id review_full_text embedding 5093 d822e1163f03e3046f92b9154ba0df2a o produto chegou bem antes da data prevista, muito bom, obrigada! List(-0.050139178, 0.036175992, -0.0056547774, -0.03589094, -0.010726606, 0.010208395, -0.06266727, 0.060949408, -0.0010032038, -0.002541577, 0.011315945, 0.009095509, 0.0109504415, -0.053059086, -0.033159062, -0.04047914, 0.084640615, -0.063812, -0.050735395, -0.069409996, -0.0035408528, 0.003948944, -0.0082507925, 0.043118488, 0.022299564, 0.01395041, 0.04118135, 0.031635925, -0.005582076, -0.08400165, -0.053071085, 0.004483497, 0.0882653, -0.004016173, -0.031743597, 0.025592174, 0.015292117, -0.04575668, -0.043327633, -0.03849557, -0.0090389, -0.031071737, -0.0027046758, 0.05833422, 0.031533882, 0.062353045, -0.005341865, 0.012450109, 0.016480444, 0.03700734, 0.008607413, -0.019250639, -0.08509968, -0.06244032, 0.027054355, 0.062263664, -0.027489347, -0.002500785, -0.03911687, 0.033685632, -0.021422671, -0.07386823, -0.09702153, 0.0661294, -0.013638461, 0.008045347, -0.03151679, -0.05698824, -0.026790338, 0.06660216, -0.02553978, 0.052405167, 0.037153725, -0.040300343, -7.166513E-4, 0.06716671, 0.072625436, -0.06743383, -0.052287444, -0.019454734, 0.020377634, 0.017021958, -0.06082211, 0.044032898, -0.011520052, 0.09570481, 0.06379171, 0.0334565, 0.034642152, -0.065300465, 0.05197593, 0.0063548256, -0.029034587, -0.009881388, -0.010906532, -0.0047931424, -0.020970885, -0.07427916, 0.054572802, 0.0898346, -0.03049673, 0.0050897696, -0.0047085383, -0.16359377, -0.0025553906, -0.0074900375, -0.0076291966, 0.053078897, 0.04400485, -0.027941216, -0.018073922, -0.009692555, -0.028276958, -0.0029445025, -0.053674944, 0.082384, -0.012959384, 0.006739205, 0.097394116, -0.075895764, 0.048696753, 0.004334864, 0.09179624, -0.060663242, -0.06733632, -0.051109467, 0.1510071, 0.009484408, -0.0533726, -0.0077202846, 0.049080912, -0.061952766, 0.016223604, 0.045544215, 0.03408416, -0.03712163, -0.054965287, 0.08574532, 0.009901143, -0.025059948, -0.03526782, 0.048047185, -0.03729791, -0.02158164, 0.009173136, -0.075023405, 0.06825107, -0.015713701, 0.07439909, -0.095079266, -0.035700154, 0.005908887, 0.08879752, 0.010976754, 0.029335009, 0.030702053, 0.034813467, 0.03853357, 0.08120003, -0.017680187, -0.01808491, -0.06322619, 0.05734152, -0.049683638, -0.08787216, -0.06975961, 0.030784577, -0.04478481, 0.08780558, 0.050544605, 0.017002543, -0.03267801, -0.1115464, 0.015987735, 0.0044044335, 0.05998722, 0.119135685, 0.0076278546, -0.0607149, 0.031245021, 0.012317424, 0.007777316, -0.017136287, -0.006838459, -0.015085494, -0.0020334425, -0.034190677, -0.02059848, 0.052771773, 0.06463149, 0.006760347, -0.055914346, 0.037685156, 0.03067777, 0.014669008, -0.027285067, 0.032473184, 0.0017112177, 0.07793201, -0.023431836, -0.11008806, -0.048461363, 0.022362098, -0.0030470367, 0.03280629, -0.011004754, 0.050851896, -0.025562424, -0.042477243, 0.016359525, 0.01583762, -0.006026602, 0.01674724, 0.031131139, -0.045645546, 0.09246585, -0.013155758, -0.053534627, -0.066256, -0.0041151056, -0.0032535745, 0.02998395, -0.021655211, -0.10037691, 0.05203766, -0.023028268, -0.02349008, 0.011234796, 0.009470106, 3.7934823E-4, -0.050792046, 0.11720002, 0.038126405, 0.10604131, 0.019138487, -6.1324023E-4, 0.026052862, -0.02774328, -5.008504E-4, 0.10792994, 0.02039427, -0.040497143, 0.05153298, -0.088846035, 0.073429845, 0.057590652, -0.06462396, -0.005156672, -0.10122218, 0.096944004, -0.010256859, 0.024311392, -0.08704607, 0.013319866, 0.035935577, -0.050785463, -0.061280154, 0.09672185, -0.047083393, 0.059229955, 0.13216336, -0.15191698, -0.060721617, -0.015851742, 0.013236694, 0.0846542, -0.0044395486, 0.020459637, -0.047889758, -0.06761571, -0.01682474, -0.008010918, 0.08599098, 0.044799563, 0.025787044, -0.023637192, -6.307329E-4, 0.055191092, -0.0149136, 0.119661875, 0.003247709, -0.025155665, 0.038600132, 0.021224871, -0.046931468, -0.03158681, 0.07920909, -0.08856252, 0.04434131, 0.0011984867, 0.0142609

## 3. Indexação e validação do Vector Search

Nesta etapa, utilizamos o **Databricks Vector Search** para indexar os embeddings armazenados no Delta Lake e permitir a recuperação semântica das avaliações.

### Configuração do índice vetorial

O índice foi criado pela interface do Databricks, com as seguintes configurações:

| Parâmetro | Configuração |
|---|---|
| Endpoint | `f4_reviews_endpoint` |
| Índice | `workspace.raw.f4_reviews_vector_index` |
| Tabela de origem | `workspace.raw.f4_reviews_embeddings` |
| Chave primária | `embedding_id` |
| Coluna vetorial | `embedding` |
| Dimensão dos vetores | 384 |
| Tipo de índice | Delta Sync — Hybrid |
| Sincronização | Triggered |
| Registros indexados | 40.739 |

A sincronização no modo **Triggered** permite atualizar o índice sob demanda, após alterações na tabela Delta.

### Validação da recuperação

Para verificar o funcionamento do banco vetorial, realizamos dois testes:

1. **Busca semântica Top-5:** recuperação das cinco avaliações mais similares a uma pergunta sobre atraso na entrega.
2. **Filtro por metadados:** aplicação de um filtro para recuperar somente avaliações com `review_score = 1`.

Ambos os testes utilizam embeddings de consulta gerados pelo mesmo modelo empregado na indexação.

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

pergunta = "Clientes reclamando de atraso na entrega"

vetor = modelo.encode(
    pergunta,
    normalize_embeddings=True
).tolist()

resultado = w.vector_search_indexes.query_index(
    index_name="workspace.raw.f4_reviews_vector_index",
    columns=["review_full_text", "review_score", "order_status"],
    query_vector=vetor,
    num_results=5,
    query_type="ANN"
)

print(resultado.as_dict())

/local_disk0/.ephemeral_nfs/envs/pythonEnv-ad6aa719-1290-430a-80bb-1e37e3de139a/lib/python3.12/site-packages/torch/_vmap_internals.py:9: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  from torch.utils._pytree import _broadcast_to_and_flatten, tree_flatten, tree_unflatten


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

{'manifest': {'column_count': 4, 'columns': [{'name': 'review_full_text'}, {'name': 'review_score'}, {'name': 'order_status'}, {'name': 'score'}]}, 'next_page_token': '', 'result': {'data_array': [['atraso na entrega', 1.0, 'delivered', 0.7271288], ['atraso na entrega', 1.0, 'delivered', 0.7271288], ['atraso na entrega', 2.0, 'delivered', 0.7271288], ['entrega em atraso', 2.0, 'delivered', 0.72538894], ['demora entrega', 3.0, 'delivered', 0.71718466]], 'row_count': 5}}


In [0]:
import json

resultado_filtrado = w.vector_search_indexes.query_index(
    index_name="workspace.raw.f4_reviews_vector_index",
    columns=["review_full_text", "review_score", "order_status"],
    query_vector=vetor,
    num_results=5,
    query_type="ANN",
    filters_json=json.dumps({"review_score": 1})
)

print(resultado_filtrado.as_dict())

{'manifest': {'column_count': 4, 'columns': [{'name': 'review_full_text'}, {'name': 'review_score'}, {'name': 'order_status'}, {'name': 'score'}]}, 'next_page_token': '', 'result': {'data_array': [['atraso na entrega', 1.0, 'delivered', 0.7271422], ['atraso na entrega', 1.0, 'delivered', 0.7271422], ['demora na entrega', 1.0, 'delivered', 0.71132517], ['demora na entrega', 1.0, 'delivered', 0.71132517], ['demora na entrega', 1.0, 'delivered', 0.71132517]], 'row_count': 5}}


## 4. Integração com RAG e conclusão

Após a geração, persistência e indexação dos embeddings, disponibilizamos as configurações necessárias para integrar o banco vetorial ao pipeline RAG desenvolvido pelo grupo.

O dicionário `CONFIG_RAG` centraliza os parâmetros de acesso ao índice, incluindo o endpoint, o modelo de embeddings, a dimensionalidade dos vetores e os metadados disponíveis para consulta.

### Orientações para integração

O componente de recuperação do RAG deverá:

1. Carregar o mesmo modelo de embeddings utilizado neste notebook.
2. Converter as perguntas dos usuários em vetores normalizados de 384 dimensões.
3. Consultar o índice `workspace.raw.f4_reviews_vector_index` utilizando o endpoint configurado.
4. Recuperar os textos relevantes e, quando necessário, aplicar filtros por metadados.

### Resultado final

Foram gerados e armazenados **40.739 embeddings**, posteriormente indexados no Databricks Vector Search.

Os testes confirmaram o funcionamento da recuperação semântica Top-5 e a aplicação de filtros por nota de avaliação.

O banco vetorial está preparado para ser utilizado como componente de recuperação de informações da solução RAG.

In [0]:
CONFIG_RAG = {
    "endpoint": "f4_reviews_endpoint",
    "index": "workspace.raw.f4_reviews_vector_index",
    "embedding_model": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    "embedding_dimension": 384,
    "text_column": "review_full_text",
    "metadata_columns": [
        "review_score",
        "order_status",
        "product_category_name_english",
        "order_purchase_timestamp"
    ],
    "sync_mode": "TRIGGERED"
}

print(CONFIG_RAG)

{'endpoint': 'f4_reviews_endpoint', 'index': 'workspace.raw.f4_reviews_vector_index', 'embedding_model': 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2', 'embedding_dimension': 384, 'text_column': 'review_full_text', 'metadata_columns': ['review_score', 'order_status', 'product_category_name_english', 'order_purchase_timestamp'], 'sync_mode': 'TRIGGERED'}


## Resultado — Embeddings e Banco Vetorial

**Modelo selecionado:** `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`

**Justificativa:** apresentou Precision@5 de 0,40 no experimento comparativo, com menor tempo de processamento entre os três modelos avaliados.

**Base processada:** 40.739 avaliações de clientes da Olist.

**Arquitetura:** embeddings normalizados de 384 dimensões, persistidos na tabela Delta `workspace.raw.f4_reviews_embeddings` e indexados no Databricks Vector Search.

**Configuração do índice:**
- Endpoint: `f4_reviews_endpoint`
- Índice: `workspace.raw.f4_reviews_vector_index`
- Tipo: Delta Sync, Hybrid
- Sincronização: Triggered
- Registros indexados: 40.739
- Status: Online

**Validação:**
- Busca semântica Top-5: executada com sucesso para reclamações de atraso na entrega.
- Filtro por metadados `review_score = 1`: executado com sucesso.
- Comparação de modelos: Precision@5 e MRR@5 calculadas sobre amostra experimental.

**Integração com RAG:** o componente de recuperação deve gerar embeddings das perguntas usando o mesmo modelo selecionado e consultar o índice por similaridade vetorial, podendo aplicar filtros por metadados.